# Statistical comparison of reinforcement-learning strategies

This notebook compares **all available strategies pairwise** and compares every strategy against the chosen **baseline**.

## Statistical plan
- **Primary outcome:** episode score / survival steps (higher is better).
- **Secondary outcomes:** mean absolute pole angle (lower is better) and shaped return (only comparable between methods using the same reward definition).
- If strategies were evaluated at the same pole lengths, the notebook uses **paired tests across pole lengths** on each method's mean score per length: paired t-test and Wilcoxon signed-rank test. The Wilcoxon result is the safer default when differences are skewed.
- Holm correction controls family-wise error across the many pairwise comparisons.
- Effect sizes and confidence intervals are reported alongside p-values.

**Important design caveat:** pole lengths are the paired units here, not independent training seeds. If you have repeated independent training seeds for each strategy, the stronger design is to compare those seed-level results. Repeated episodes from one trained model do not measure training-to-training variability. Do not treat the 300 episodes as 300 independent models.

The ZIP currently includes run-level CSVs for reward shaping, heuristic, and random strategies. It does not appear to include raw per-run evaluation results for every DQN variant. The notebook will load available files and report only comparisons supported by run-level data. Add your baseline/other strategy CSVs to `results/` to include them.

## 1. Imports and configuration

Expected CSV format: one row per evaluation episode, with at least `pole_length` and `score`. Optional columns include `seed`, `run`, `episode`, `mean_abs_angle`, and `shaped_return`.

Edit `STRATEGY_FILES` to match your filenames. Paths are relative to this notebook's folder. For files inside a `results/` folder, keep that folder alongside the notebook.

In [ ]:
from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display

RESULTS_DIR = Path("results")

# Strategy label: path to per-episode/per-run CSV.
# Add every strategy you want to compare. The baseline label below is used
# for the explicit strategy-vs-baseline table.
BASELINE_NAME = "baseline"

STRATEGY_FILES = {
    "reward_shaping": RESULTS_DIR / "test_runs_reward_shaping.csv",
    "heuristic": RESULTS_DIR / "heuristic_episodes.csv",
    "random": RESULTS_DIR / "random_episodes.csv",
    # Uncomment/edit these when you have run-level results:
    # "baseline": RESULTS_DIR / "test_runs_baseline.csv",
    # "symmetry": RESULTS_DIR / "test_runs_symmetry.csv",
    # "disturbance": RESULTS_DIR / "test_runs_disturbance.csv",
}

ALPHA = 0.05
N_BOOTSTRAP = 5000
RANDOM_STATE = 42
PRIMARY_METRIC = "score"  # higher is better


## 2. Load and standardize the data

In [ ]:
def load_strategy(name, path):
    path = Path(path)
    if not path.exists():
        warnings.warn(f"Skipping {name}: file not found at {path}")
        return None
    df = pd.read_csv(path).copy()

    # Harmonize common score-column names.
    if "score" not in df.columns:
        for candidate in ("mean_score", "episode_steps", "steps"):
            if candidate in df.columns:
                df = df.rename(columns={candidate: "score"})
                break
    required = {"pole_length", "score"}
    missing = required - set(df.columns)
    if missing:
        warnings.warn(f"Skipping {name}: missing columns {sorted(missing)}")
        return None

    df["pole_length"] = pd.to_numeric(df["pole_length"], errors="coerce")
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df = df.dropna(subset=["pole_length", "score"])
    df["strategy"] = name
    if "seed" not in df.columns:
        df["seed"] = np.nan
    if "run" not in df.columns:
        if "episode" in df.columns:
            df["run"] = df["episode"]
        else:
            df["run"] = df.groupby("pole_length").cumcount() + 1
    return df

loaded = {}
for name, path in STRATEGY_FILES.items():
    frame = load_strategy(name, path)
    if frame is not None and not frame.empty:
        loaded[name] = frame

if len(loaded) < 2:
    raise ValueError("Need at least two readable strategy CSV files to compare.")

all_runs = pd.concat(loaded.values(), ignore_index=True)
print("Strategies loaded:", list(loaded))
display(pd.DataFrame([
    {"strategy": name, "episodes": len(df),
     "pole_lengths": df["pole_length"].nunique(),
     "mean_score": df["score"].mean(),
     "sd_score": df["score"].std()}
    for name, df in loaded.items()
]).sort_values("strategy").reset_index(drop=True))


## 3. Descriptive statistics and visualization

Scores are summarized both overall and by pole length. Overall averages are descriptive; if strategies were tested on different length distributions, use the shared-length comparisons below.

In [ ]:
summary_rows = []
for name, df in loaded.items():
    row = {
        "strategy": name,
        "n_episodes": len(df),
        "n_pole_lengths": df["pole_length"].nunique(),
        "mean_score": df["score"].mean(),
        "sd_score": df["score"].std(),
        "median_score": df["score"].median(),
        "min_score": df["score"].min(),
        "max_score": df["score"].max(),
    }
    for metric in ("mean_abs_angle", "shaped_return"):
        if metric in df:
            row[f"mean_{metric}"] = pd.to_numeric(df[metric], errors="coerce").mean()
    summary_rows.append(row)
descriptive = pd.DataFrame(summary_rows).sort_values("strategy")
display(descriptive.round(4))

length_means = (all_runs.groupby(["strategy", "pole_length"], as_index=False)
                .agg(mean_score=("score", "mean"), sd_score=("score", "std"),
                     n=("score", "size")))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(11, 5))
    for name, part in length_means.groupby("strategy"):
        plt.plot(part["pole_length"], part["mean_score"], marker="o", label=name)
    plt.xlabel("Pole length")
    plt.ylabel("Mean score")
    plt.title("Mean evaluation score by pole length")
    plt.legend()
    plt.grid(alpha=.25)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib to display plots.")


## 4. Statistical tests across strategies

**Primary analysis:** for each pair of strategies, aggregate repeated evaluation episodes to a mean score for each pole length, then compare those means on the pole lengths both strategies share. This controls for pole length and avoids pretending individual episodes from a single trained agent are independent model replications.

- Paired t-test: tests whether the mean paired difference is zero.
- Wilcoxon signed-rank: non-parametric paired test.
- Effect size: mean difference (A − B) and paired Cohen's \(d_z\).
- 95% bootstrap CI for mean paired difference, resampling pole lengths.
- Holm-adjusted p-values are computed across all pairwise Wilcoxon tests.

If there are fewer than 5 shared pole lengths, interpret tests cautiously. If there are zero differences or too few nonzero differences, Wilcoxon may be undefined; the code records that result instead of crashing.

In [ ]:
def bootstrap_mean_ci(differences, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    x = np.asarray(differences, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    samples = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return tuple(np.quantile(samples, [0.025, 0.975]))

def paired_comparison(name_a, name_b, metric="score"):
    a = loaded[name_a].groupby("pole_length")[metric].mean()
    b = loaded[name_b].groupby("pole_length")[metric].mean()
    paired = pd.concat([a.rename("A"), b.rename("B")], axis=1).dropna()
    x, y = paired["A"].to_numpy(), paired["B"].to_numpy()
    diff = x - y
    n = len(diff)
    result = {
        "strategy_A": name_a, "strategy_B": name_b, "metric": metric,
        "n_shared_lengths": n,
        "mean_A": np.mean(x) if n else np.nan,
        "mean_B": np.mean(y) if n else np.nan,
        "mean_difference_A_minus_B": np.mean(diff) if n else np.nan,
        "median_difference": np.median(diff) if n else np.nan,
        "ci95_low": np.nan, "ci95_high": np.nan,
        "paired_t_stat": np.nan, "paired_t_p": np.nan,
        "wilcoxon_stat": np.nan, "wilcoxon_p": np.nan,
        "cohens_dz": np.nan,
        "direction": "A higher" if n and np.mean(diff) > 0 else ("B higher" if n and np.mean(diff) < 0 else "tie")
    }
    if n >= 2:
        result["ci95_low"], result["ci95_high"] = bootstrap_mean_ci(diff)
        sd = np.std(diff, ddof=1)
        result["cohens_dz"] = np.mean(diff) / sd if sd > 0 else np.nan
        try:
            t = stats.ttest_rel(x, y, nan_policy="omit")
            result["paired_t_stat"], result["paired_t_p"] = t.statistic, t.pvalue
        except Exception:
            pass
        try:
            if np.any(np.abs(diff) > 1e-12):
                w = stats.wilcoxon(x, y, alternative="two-sided", method="auto")
                result["wilcoxon_stat"], result["wilcoxon_p"] = w.statistic, w.pvalue
        except (ValueError, ZeroDivisionError):
            pass
    return result

strategy_names = list(loaded)
pair_rows = []
for a, b in itertools.combinations(strategy_names, 2):
    pair_rows.append(paired_comparison(a, b, PRIMARY_METRIC))

pairwise = pd.DataFrame(pair_rows)
valid = pairwise["wilcoxon_p"].notna()
pairwise["wilcoxon_p_holm"] = np.nan
if valid.any():
    pairwise.loc[valid, "wilcoxon_p_holm"] = multipletests(
        pairwise.loc[valid, "wilcoxon_p"], alpha=ALPHA, method="holm"
    )[1]
pairwise["significant_holm"] = pairwise["wilcoxon_p_holm"] < ALPHA
display(pairwise.sort_values("wilcoxon_p_holm", na_position="last").round(5))


## 5. Compare every strategy against the baseline

Set `BASELINE_NAME` in Section 1 to the exact label used in `STRATEGY_FILES`. This section runs the same paired tests for each other strategy against that baseline and corrects for multiple comparisons with Holm's method.

In [ ]:
if BASELINE_NAME not in loaded:
    print(
        f"Baseline '{BASELINE_NAME}' is not loaded. Add its run-level CSV to "
        "STRATEGY_FILES and set BASELINE_NAME to that dictionary key. "
        "No baseline-specific tests were run."
    )
    baseline_comparisons = pd.DataFrame()
else:
    rows = [
        paired_comparison(name, BASELINE_NAME, PRIMARY_METRIC)
        for name in strategy_names if name != BASELINE_NAME
    ]
    baseline_comparisons = pd.DataFrame(rows)
    valid = baseline_comparisons["wilcoxon_p"].notna()
    baseline_comparisons["wilcoxon_p_holm"] = np.nan
    if valid.any():
        baseline_comparisons.loc[valid, "wilcoxon_p_holm"] = multipletests(
            baseline_comparisons.loc[valid, "wilcoxon_p"], alpha=ALPHA, method="holm"
        )[1]
    baseline_comparisons["significant_holm"] = baseline_comparisons["wilcoxon_p_holm"] < ALPHA
    display(baseline_comparisons.round(5))


## 6. Optional secondary outcome tests

Run the same pairwise comparison for `mean_abs_angle` (lower is better) when all strategies contain that column. Do **not** compare `shaped_return` across methods if they use different reward functions, because the numerical scales are not equivalent.

In [ ]:
for metric in ("mean_abs_angle",):
    eligible = [name for name, df in loaded.items() if metric in df.columns]
    if len(eligible) >= 2:
        rows = [paired_comparison(a, b, metric)
                for a, b in itertools.combinations(eligible, 2)]
        result = pd.DataFrame(rows)
        valid = result["wilcoxon_p"].notna()
        result["wilcoxon_p_holm"] = np.nan
        if valid.any():
            result.loc[valid, "wilcoxon_p_holm"] = multipletests(
                result.loc[valid, "wilcoxon_p"], alpha=ALPHA, method="holm"
            )[1]
        print(f"Secondary outcome: {metric} (lower is better)")
        display(result.round(5))
    else:
        print(f"Skipping {metric}: fewer than two strategies have this metric.")


## 7. Export tables and write-up checklist

The CSV files contain the numerical results for your report. Avoid claiming that a strategy is universally better based only on statistical significance. Report the effect size, confidence interval, adjusted p-value, shared pole lengths, and experimental limitations.

In [ ]:
OUTPUT_DIR = Path("statistical_test_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

descriptive.to_csv(OUTPUT_DIR / "descriptive_statistics.csv", index=False)
pairwise.to_csv(OUTPUT_DIR / "all_pairwise_comparisons.csv", index=False)
if not baseline_comparisons.empty:
    baseline_comparisons.to_csv(OUTPUT_DIR / "baseline_comparisons.csv", index=False)

print(f"Saved tables to: {OUTPUT_DIR.resolve()}")
print("\nReporting checklist:")
print("1. State primary metric and why higher/lower is better.")
print("2. Report mean ± SD per strategy and performance across shared pole lengths.")
print("3. Report Wilcoxon statistic, raw p, Holm-adjusted p, mean difference, 95% CI, and Cohen's dz.")
print("4. Describe number of pole lengths and whether trained models had independent training seeds.")
print("5. If only one trained model per strategy was evaluated, limit conclusions to these trained models.")
